# Dataset Building — BLE Indoor Localization

Este notebook documenta cómo se construye el dataset de entrenamiento para el sistema de
localización BLE indoor. Cubre los tres componentes principales:

1. **Entorno físico** — habitación, gateways, zonas espaciales
2. **Simuladores RSSI** — PathLoss (analítico) y Sionna RT (ray tracing)
3. **Generación de trayectorias** — random walk que produce el CSV de entrenamiento

El output final es un CSV con columnas `session_id, x_m, y_m, zone_id, zone_name, n_visible, rssi_A1..A4`
que alimenta directamente el pipeline de entrenamiento (`python -m ble_indoor train`).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "baseline_room.yaml").is_file():
    REPO_ROOT = Path.cwd().parent
if not (REPO_ROOT / "config" / "baseline_room.yaml").is_file():
    raise FileNotFoundError("Ejecuta el notebook desde la raíz del repo.")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ble_indoor.settings import ProjectConfig, ProjectLayout

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

layout = ProjectLayout(REPO_ROOT)
cfg = ProjectConfig.load(layout.default_config_path())
env = cfg.environment

print("Repo:", REPO_ROOT)
print(f"Habitación: {env.room.width_m}m × {env.room.height_m}m")
print(f"Gateways: {list(env.gateway_ids)}")
print(f"Zonas: {cfg.spatial_zones.nx}×{cfg.spatial_zones.ny} = {cfg.spatial_zones.n_zones}")

## 1. Entorno físico

El entorno está definido en `config/baseline_room.yaml`. Es una habitación rectangular con
4 gateways BLE fijos en las esquinas y una grilla de zonas espaciales superpuesta.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

room_w = env.room.width_m
room_h = env.room.height_m
nx, ny = cfg.spatial_zones.nx, cfg.spatial_zones.ny
cell_w = room_w / nx
cell_h = room_h / ny

# Zona grid
colors = plt.cm.Pastel1(np.linspace(0, 0.8, nx * ny))
for iy in range(ny):
    for ix in range(nx):
        zone_id = iy * nx + ix
        rect = mpatches.FancyBboxPatch(
            (ix * cell_w, iy * cell_h), cell_w, cell_h,
            boxstyle="square,pad=0",
            linewidth=1.2, edgecolor="gray",
            facecolor=colors[zone_id], alpha=0.6,
        )
        ax.add_patch(rect)
        ax.text(
            ix * cell_w + cell_w / 2, iy * cell_h + cell_h / 2,
            f"Z{zone_id}\n({cfg.spatial_zones.zone_name(zone_id)})",
            ha="center", va="center", fontsize=9, color="#333",
        )

# Gateways
gw_pos = env.gateway_positions_m()
ax.scatter(gw_pos[:, 0], gw_pos[:, 1], c="crimson", s=160, zorder=5, marker="^", label="Gateway")
for gw in env.gateways:
    offset_x = 0.3 if gw.x_m < room_w / 2 else -0.3
    offset_y = 0.35 if gw.y_m < room_h / 2 else -0.35
    ax.text(gw.x_m + offset_x, gw.y_m + offset_y, gw.id,
            fontsize=9, fontweight="bold", color="crimson",
            ha="center", va="center")

ax.set_xlim(0, room_w)
ax.set_ylim(0, room_h)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(f"Entorno: {room_w}m × {room_h}m — {cfg.spatial_zones.n_zones} zonas — 4 gateways BLE")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 2. Simuladores RSSI

El sistema soporta dos simuladores intercambiables — ambos implementan el mismo protocolo
`RssiObservationSource` con tres métodos:

```
mean_rssi_dbm(position)              → RSSI medio por gateway (sin ruido)
sample_rssi_dbm(position, rng)       → un draw con ruido gaussiano
sample_rssi_with_reception(pos, rng) → draw + dropout probabilístico por gateway
```

### 2a. PathLossSimulator (analítico)

Modelo log-distancia clásico:

$$\text{RSSI}(d) = P_{tx} - 10 \cdot n \cdot \log_{10}(d) + \mathcal{N}(0, \sigma^2)$$

- $n = 2.2$ (exponente de path loss — interior con paredes)
- $\sigma = 3$ dB (ruido de shadowing)
- No modela reflexiones, difracción ni obstáculos

In [ ]:
from ble_indoor.simulation.path_loss import PathLossSimulator

sim_pl = PathLossSimulator(env)

# Mapa de RSSI medio para cada gateway sobre una grilla fina
res = 0.15
xs = np.arange(res / 2, room_w, res)
ys = np.arange(res / 2, room_h, res)
grid = np.array(np.meshgrid(xs, ys, indexing="ij")).reshape(2, -1).T

rssi_pl = np.array([sim_pl.mean_rssi_dbm(p) for p in grid])  # (N, 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col_idx, gw in zip(axes, range(4), env.gateways):
    z = rssi_pl[:, col_idx].reshape(len(xs), len(ys))
    im = ax.imshow(
        z.T, origin="lower", aspect="auto",
        extent=[0, room_w, 0, room_h],
        cmap="RdYlGn", vmin=-100, vmax=-50,
    )
    ax.scatter([gw.x_m], [gw.y_m], c="white", s=100, marker="^", zorder=5)
    ax.set_title(f"{gw.id} (PathLoss)")
    ax.set_xlabel("x (m)")
    if col_idx == 0:
        ax.set_ylabel("y (m)")
    plt.colorbar(im, ax=ax, label="dBm")

plt.suptitle("RSSI medio por gateway — PathLossSimulator", y=1.02)
plt.tight_layout()
plt.show()

### 2b. SionnaRTSimulator (ray tracing)

Usa [Sionna RT](https://nvlabs.github.io/sionna/api/rt.html) de NVIDIA para simular propagación
de radio con ray tracing físico. A diferencia del path loss analítico, modela:

- **Reflexiones** en paredes, suelo y techo
- **Propiedades dieléctricas** del material (concreto: $\varepsilon_r = 5.24$, $\sigma = 0.014$ S/m)
- **Interferencia constructiva/destructiva** entre múltiples trayectorias
- **Frecuencia portadora** BLE: 2.4 GHz

La ganancia de canal se convierte a RSSI como:
$$\text{RSSI} = P_{tx} + 10 \cdot \log_{10}\left(\sum_i |a_i|^2\right) \quad [\text{dBm}]$$

**Estrategia de precomputo + caché:**

```
SionnaRTSimulator.__init__()
  └─ Precomputa grilla 0.25m (~1500 puntos) × 4 gateways  ← solo la 1ª vez
       ├─ Construye escena Mitsuba3 XML (habitación 3D)
       ├─ Corre PathSolver de Sionna RT (GPU/CPU)
       └─ Guarda → data/simulated/sionna_rt_cache.npz
  └─ Corridas siguientes → carga .npz (instantáneo)
  └─ Queries → KD-tree interpolation sobre la grilla
```

**Instalación requerida:**
```bash
pip install -r requirements-sionna.txt
```

In [ ]:
# Cargar SionnaRTSimulator (requiere: pip install -r requirements-sionna.txt)
# Si no está instalado, esta celda imprime un aviso y no falla el resto del notebook.

sim_srt = None
try:
    from ble_indoor.simulation.sionna_rt_simulator import SionnaRTSimulator
    sim_srt = SionnaRTSimulator(env, cfg.sionna_rt, layout_root=REPO_ROOT)
    print("SionnaRTSimulator listo.")
except ImportError:
    print("Sionna RT no instalado. Ejecutar: pip install -r requirements-sionna.txt")
    print("Las celdas de Sionna se saltarán.")

In [ ]:
if sim_srt is not None:
    rssi_srt = np.array([sim_srt.mean_rssi_dbm(p) for p in grid])  # (N, 4)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, col_idx, gw in zip(axes, range(4), env.gateways):
        z = rssi_srt[:, col_idx].reshape(len(xs), len(ys))
        im = ax.imshow(
            z.T, origin="lower", aspect="auto",
            extent=[0, room_w, 0, room_h],
            cmap="RdYlGn", vmin=-100, vmax=-50,
        )
        ax.scatter([gw.x_m], [gw.y_m], c="white", s=100, marker="^", zorder=5)
        ax.set_title(f"{gw.id} (Sionna RT)")
        ax.set_xlabel("x (m)")
        if col_idx == 0:
            ax.set_ylabel("y (m)")
        plt.colorbar(im, ax=ax, label="dBm")

    plt.suptitle("RSSI medio por gateway — SionnaRTSimulator (ray tracing)", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("(celda omitida — Sionna RT no disponible)")

### 2c. Comparación PathLoss vs Sionna RT

El path loss produce mapas suaves y simétricos. Sionna RT introduce variaciones espaciales
por interferencia multitrayecto — zonas de alta señal donde las reflexiones se suman
constructivamente, y zonas de baja señal por interferencia destructiva.

In [ ]:
if sim_srt is not None:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, col_idx, gw in zip(axes, range(4), env.gateways):
        diff = (rssi_srt[:, col_idx] - rssi_pl[:, col_idx]).reshape(len(xs), len(ys))
        lim = np.percentile(np.abs(diff), 95)
        im = ax.imshow(
            diff.T, origin="lower", aspect="auto",
            extent=[0, room_w, 0, room_h],
            cmap="RdBu_r", vmin=-lim, vmax=lim,
        )
        ax.scatter([gw.x_m], [gw.y_m], c="black", s=100, marker="^", zorder=5)
        ax.set_title(f"{gw.id}: Sionna − PathLoss")
        ax.set_xlabel("x (m)")
        if col_idx == 0:
            ax.set_ylabel("y (m)")
        plt.colorbar(im, ax=ax, label="ΔdBm")

    plt.suptitle("Diferencia RSSI (Sionna RT − PathLoss) — rojo: Sionna mayor, azul: PathLoss mayor", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("(celda omitida — Sionna RT no disponible)")

## 3. Generación del dataset de trayectorias

`TrajectoryFingerprintDatasetBuilder` simula un badge BLE moviéndose por la habitación
y grabando lecturas RSSI en cada paso.

**Parámetros clave** (de `config/baseline_room.yaml`):

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `n_sessions` | 14 | sesiones independientes de movimiento |
| `steps_per_session` | 320 | pasos por sesión |
| `step_m` | 0.35 m | largo de cada paso |
| `brownian_std_m` | 0.1 m | ruido de movimiento (Browniano) |
| `gateway_reception_prob` | 0.86 | probabilidad de recibir cada gateway |
| `min_visible_gateways` | 3 | mínimo de gateways visibles para registrar la muestra |
| `missing_rssi_dbm` | −105 dBm | valor para gateways no recibidos |

**Proceso por paso:**
```
posición_nueva = posición + dirección_aleatoria * step_m + N(0, brownian_std)
RSSI, visible  = simulator.sample_rssi_with_reception(pos, rng, prob=0.86, missing=-105)
si sum(visible) >= 3: registrar fila
```

In [ ]:
from ble_indoor.data.builders import TrajectoryFingerprintDatasetBuilder
from ble_indoor.simulation.path_loss import PathLossSimulator

# Demostración con PathLoss (no requiere Sionna)
sim_demo = PathLossSimulator(env)
builder = TrajectoryFingerprintDatasetBuilder(
    env, cfg.trajectory, cfg.spatial_zones, sim_demo
)
df_demo = builder.build_dataframe()

print(f"Dataset generado: {len(df_demo)} filas × {len(df_demo.columns)} columnas")
print("Columnas:", list(df_demo.columns))
df_demo.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Trayectorias por sesión
ax = axes[0]
cmap = plt.cm.tab20
for sid, grp in df_demo.groupby("session_id"):
    ax.plot(grp["x_m"], grp["y_m"], lw=0.7, alpha=0.6,
            color=cmap(int(sid) / cfg.trajectory.n_sessions))
gw_pos = env.gateway_positions_m()
ax.scatter(gw_pos[:, 0], gw_pos[:, 1], c="red", s=120, marker="^", zorder=5, label="GW")
ax.set_xlim(0, room_w)
ax.set_ylim(0, room_h)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(f"{cfg.trajectory.n_sessions} trayectorias de random walk")
ax.legend()

# Distribución de muestras por zona
ax2 = axes[1]
zone_counts = df_demo["zone_id"].value_counts().sort_index()
zone_labels = [cfg.spatial_zones.zone_name(z) for z in zone_counts.index]
ax2.bar(zone_labels, zone_counts.values, color="steelblue", edgecolor="white")
ax2.set_xlabel("Zona")
ax2.set_ylabel("Muestras")
ax2.set_title("Muestras por zona (balance de clases)")

plt.tight_layout()
plt.show()

## 4. Distribución RSSI en el dataset

In [ ]:
rssi_cols = [f"rssi_{g}" for g in env.gateway_ids]
missing_dbm = cfg.trajectory.missing_rssi_dbm

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, rssi_cols):
    received = df_demo[col][df_demo[col] > missing_dbm + 0.5]
    ax.hist(received, bins=40, color="teal", alpha=0.8, edgecolor="none")
    frac_missing = (df_demo[col] <= missing_dbm + 0.5).mean()
    ax.set_title(f"{col}\nmissing: {frac_missing:.0%}")
    ax.set_xlabel("dBm")
axes[0].set_ylabel("Frecuencia")

plt.suptitle("Distribución RSSI recibido por gateway (excl. missing)", y=1.02)
plt.tight_layout()
plt.show()

## 5. Generar el CSV de entrenamiento

El CSV se puede generar desde la CLI o directamente desde Python.

In [ ]:
# Opción A — PathLoss (rápido, sin dependencias extra)
# PYTHONPATH=src python -m ble_indoor generate-csv --simulator pathloss --force

# Opción B — Sionna RT (ray tracing, requiere: pip install -r requirements-sionna.txt)
# PYTHONPATH=src python -m ble_indoor generate-csv --simulator sionna --force

# Desde Python directamente:
from ble_indoor.pipelines.baseline_study import BaselineStudy

study = BaselineStudy(layout)
out_path = study.train_dataset_csv_path()
print("CSV de entrenamiento:", out_path)
print("Existe:", out_path.is_file())

# Para regenerar con PathLoss:
# df = study.generate_base_dataset("trajectory", save=True)

# Para regenerar con Sionna RT (si está instalado):
# from ble_indoor.simulation.sionna_rt_simulator import SionnaRTSimulator
# src = SionnaRTSimulator(env, cfg.sionna_rt, layout_root=REPO_ROOT)
# df = study.generate_base_dataset("trajectory", save=True, rssi_source=src)

In [ ]:
# Cargar y mostrar el CSV guardado
from ble_indoor.simulation.trace_loader import load_training_trace

if out_path.is_file():
    df_csv = load_training_trace(out_path, env, cfg.spatial_zones)
    print(f"{len(df_csv)} filas, {len(df_csv.columns)} columnas")
    print("\nEstadísticas RSSI:")
    display(df_csv[[f"rssi_{g}" for g in env.gateway_ids]].describe().round(1))
else:
    print(f"No existe CSV en {out_path}")
    print("Ejecutar: PYTHONPATH=src python -m ble_indoor generate-csv --force")

## 6. Schema del CSV

El CSV de entrenamiento tiene el siguiente schema (compatible con cualquier simulador):

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `session_id` | int | ID de trayectoria (0 … n_sessions−1) |
| `x_m` | float | posición X del badge (metros) |
| `y_m` | float | posición Y del badge (metros) |
| `zone_id` | int | zona espacial (0 … n_zones−1) |
| `zone_name` | str | nombre legible (e.g. `Z0_2`) |
| `n_visible` | int | número de gateways recibidos en este paso |
| `rssi_A1` | float | RSSI gateway A1 (dBm), −105 si no recibido |
| `rssi_A2` | float | RSSI gateway A2 (dBm), −105 si no recibido |
| `rssi_A3` | float | RSSI gateway A3 (dBm), −105 si no recibido |
| `rssi_A4` | float | RSSI gateway A4 (dBm), −105 si no recibido |

Este CSV es consumido directamente por `python -m ble_indoor train --model [knn|rf]`.